# UP02: Transformed-outcome uplift vs T-learner

Question: does one model on the transformed outcome `Z = Y·T/p − Y·(1−T)/(1−p)` match or beat the two-model T-learner on the same test split?

Data: Criteo uplift dataset from the Kaggle slug `arashnic/uplift-modeling`, read in place from `/kaggle/input/uplift-modeling/`. Propensity `p` is the treated share in the training split, which is the randomized assignment rate.

Run with **Run All** inside a Kaggle Notebook with the dataset attached. This notebook was not executed locally because the data is not on this machine.

In [ ]:
import glob
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path("../src").resolve()))
from uplift_transform.data import FEATURES, OUTCOME, TREATMENT, load_criteo, stratified_split
from uplift_transform.metrics import qini_coefficient, qini_curve
from uplift_transform.tlearner import TLearner
from uplift_transform.transform import ClassTransformation

RANDOM_STATE = 42
SAMPLE_N = None  # e.g. 1_000_000 for a faster first run; None uses all rows
DATASET = "Criteo uplift (Kaggle arashnic/uplift-modeling)"
FIG_DIR = Path("../reports/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
csv_path = sorted(glob.glob("/kaggle/input/uplift-modeling/*.csv"))[0]
df = load_criteo(csv_path, sample_n=SAMPLE_N)
train, test = stratified_split(df, test_size=0.3)
print(csv_path, df.shape, "| train treated share:", round(train[TREATMENT].mean(), 4))

In [ ]:
ct = ClassTransformation().fit(train[FEATURES], train[TREATMENT], train[OUTCOME])
tl = TLearner().fit(train[FEATURES], train[TREATMENT], train[OUTCOME])

test = test.assign(
    ct_score=ct.predict_uplift(test[FEATURES]),
    tl_score=tl.predict_uplift(test[FEATURES]),
    random_score=np.random.default_rng(RANDOM_STATE).random(len(test)),
)

In [ ]:
candidates = {
    "Transformed outcome (HistGradientBoosting)": "ct_score",
    "T-learner (logistic)": "tl_score",
    "Random targeting": "random_score",
}
summary = pd.DataFrame(
    [
        {
            "model": name,
            "split": "test 30%",
            "qini_coefficient": qini_coefficient(test[OUTCOME], test[TREATMENT], test[col]),
        }
        for name, col in candidates.items()
    ]
)
summary.to_csv("../reports/up02_comparison.csv", index=False)
print(summary.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for name, col in candidates.items():
    x, gain = qini_curve(test[OUTCOME], test[TREATMENT], test[col])
    ax.plot(x, gain, label=name, linestyle="--" if col == "random_score" else "-")
ax.set(
    xlabel="Fraction of test population targeted",
    ylabel="Cumulative incremental conversions",
    title=f"Qini curves | {DATASET}\nTest split (30%, stratified by treatment) | UP02 vs UP01",
)
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "up02_qini_comparison.png", dpi=150)
plt.show()